# What a feature block adds over what the decoder already reported

A decoder produces more than an answer. It produces a weight, often a gap, sometimes a posterior.
The question this notebook measures is narrow: given all of that, plus the coarsest counts of the
record, how much more of the decoder's own failures can be anticipated from some other quantity
computed from the same record.

The answer comes back in bits per shot, against a floor. It is a comparison with one decoder's own
summary on one record. It is not a decoder benchmark and it is not a verdict.

Everything here is made inside the notebook. The last section runs against a real record if one is
on the machine, and says so politely if not.

## 1. A record and a decoder

In [1]:
import numpy as np

from qb_compiler.record import RecordSpec, residual
from qb_compiler.record.dem import build_repetition_dem, decode_records

d, rounds, shots, p = 3, 5, 8000, 0.06
rng = np.random.default_rng(3)

state = np.zeros((shots, d), dtype=np.uint8)
stored = np.zeros((shots, rounds, d - 1), dtype=np.uint8)
for t in range(rounds):
    state ^= (rng.random((shots, d)) < p).astype(np.uint8)
    parity = state[:, :-1] ^ state[:, 1:]
    stored[:, t] = parity ^ (rng.random((shots, d - 1)) < p).astype(np.uint8)
state ^= (rng.random((shots, d)) < p).astype(np.uint8)
final_parity = (state[:, :-1] ^ state[:, 1:]).astype(np.uint8)

detectors = np.empty((shots, rounds + 1, d - 1), dtype=np.uint8)
detectors[:, 0] = stored[:, 0]
detectors[:, 1:rounds] = stored[:, 1:] ^ stored[:, :-1]
detectors[:, rounds] = final_parity ^ stored[:, rounds - 1]
labels = state[:, 0].astype(np.uint8)

dem = build_repetition_dem(d, rounds, p_data=p, p_meas=p)
plain = RecordSpec(detectors=detectors, labels=labels, dem=dem)
first_pass = decode_records(dem, plain.detector_matrix())
print(
    f"shots {shots}, logical error rate {(first_pass != labels).mean():.4f}, "
    f"failures {int((first_pass != labels).sum())}"
)

shots 8000, logical error rate 0.1360, failures 1088


## 2. Something the decoder does not use

A real record can carry an association the error model does not describe. Here one site fires more
often on the shots this decoder gets wrong. The decoder reads that site like any other, because
its model says it is ordinary.

In [2]:
# One site fires more often on the shots this decoder gets wrong. The decoder sees the site, but
# its model says the site is ordinary, so it does not use the association.
tell = detectors.copy()
lit = (first_pass != labels) & (rng.random(shots) < 0.5)
tell[lit, 2, 1] ^= 1

spec = RecordSpec(detectors=tell, labels=labels, dem=dem)
served = decode_records(dem, spec.detector_matrix())
print(
    f"logical error rate {(served != labels).mean():.4f}, failures {int((served != labels).sum())}"
)

logical error rate 0.1245, failures 996


## 3. The measurement, against two floors

A held out difference of zero is not what an uninformative block scores: extra columns move the
number around and folds are finite. So the same measurement runs on records whose structure has
been destroyed and whose counts have not. Geometry nulls shuffle detector events within each round
and recompute the features. Permutation nulls reorder the feature rows against the shots.

`above_floor` compares the residual against the 95th percentile of whichever nulls ran.

In [3]:
def site_feature(record):
    """Events at one site in one round. A function of the record, so both nulls can run."""
    events = np.asarray(record.detectors, dtype=np.float64)
    return events[:, 2, 1][:, None], ["events_at_site_1_round_2"]


def unrelated(record):
    """Noise with no connection to the record at all."""
    generator = np.random.default_rng(7)
    return generator.normal(size=(record.n_shots, 1)), ["unrelated_noise"]


def show(name, report):
    print(name)
    print(f"  residual          {report.residual_bits:+.5f} bits per shot")
    print(
        f"  geometry floor    {report.nulls['geometry']['p95']:+.5f} at the 95th percentile "
        f"of {report.nulls['geometry']['n']} draws"
    )
    print(
        f"  permutation floor {report.nulls['shot_permutation']['p95']:+.5f} at the 95th "
        f"percentile of {report.nulls['shot_permutation']['n']} draws"
    )
    print(f"  pooled floor      {report.null_floor_p95:+.5f}")
    print(f"  above floor       {report.above_floor}")
    print(f"  area under curve  {report.auc_baseline:.4f} to {report.auc_augmented:.4f}")
    print()


carries = residual(spec, {"prediction": served}, site_feature, holdout="shot", n_nulls=20, seed=0)
carries_nothing = residual(
    spec, {"prediction": served}, unrelated, holdout="shot", n_nulls=20, seed=0
)
show("the site that fires on failures", carries)
show("noise", carries_nothing)

the site that fires on failures
  residual          +0.00958 bits per shot
  geometry floor    +0.00022 at the 95th percentile of 20 draws
  permutation floor +0.00026 at the 95th percentile of 20 draws
  pooled floor      +0.00026
  above floor       True
  area under curve  0.7795 to 0.7882

noise
  residual          -0.00044 bits per shot
  geometry floor    -0.00044 at the 95th percentile of 20 draws
  permutation floor +0.00017 at the 95th percentile of 20 draws
  pooled floor      +0.00002
  above floor       False
  area under curve  0.7795 to 0.7788



The report carries what it was measured against, so nothing has to be taken on trust.

In [4]:
print("target: whether the decoder's answer differs from the truth")
print(f"failures {carries.n_failures} of {carries.n_shots} shots")
print()
print("baseline columns, all of them the decoder's own output or the coarsest counts:")
for name in carries.baseline_names:
    print("  ", name)
print()
print("hold out:", carries.holdout, "with", carries.n_folds, "folds; seed", carries.seed)
for note in carries.warnings:
    print("note:", note)

target: whether the decoder's answer differs from the truth
failures 996 of 8000 shots

baseline columns, all of them the decoder's own output or the coarsest counts:
   prediction
   total_events
   events_round_0
   events_round_1
   events_round_2
   events_round_3
   events_round_4
   events_round_5

hold out: shot with 5 folds; seed 0
note: folds are stratified over shots, not over groups. Anything that drifts within a run is present on both sides of every split


## 4. Why the features are a function, not an array

Geometry nulls need the features recomputed on a shuffled record. Handing in a finished matrix
leaves nothing to recompute, so that null does not run and the report says so rather than quietly
dropping to one floor.

In [5]:
matrix = np.asarray(spec.detectors, dtype=np.float64)[:, 2, 1][:, None]
from_matrix = residual(spec, {"prediction": served}, matrix, holdout="shot", n_nulls=20, seed=0)
print(f"residual {from_matrix.residual_bits:+.5f}, above floor {from_matrix.above_floor}")
print("geometry null available:", from_matrix.nulls["geometry"]["available"])
print("reason:", from_matrix.nulls["geometry"]["reason"])

residual +0.00958, above floor True
geometry null available: False
reason: features were supplied as a matrix, so they cannot be recomputed on a shuffled record. Pass a callable to get this null


## 5. Optional: a real record

This section runs when `QB_RECORD_DATA` or `QB_QUERA_ZIP` points at the published QuEra surface
code archive and `QB_QUERA_VENDOR` points at the decoding framework published with it. It rebuilds
the published error model, decodes the published detectors with an ordinary matching decoder, and
measures what the confidence value shipped in the archive adds over that decoder's own summary.

Dataset: QuEra Computing, surface code dataset, Zenodo record 15685795. Nothing is redistributed
here.


In [6]:
import io
import os
import sys
import zipfile
from pathlib import Path

record_data = os.environ.get("QB_RECORD_DATA")
vendor_dir = os.environ.get("QB_QUERA_VENDOR")
archive = os.environ.get("QB_QUERA_ZIP")
if archive is None and record_data:
    candidate = Path(record_data) / "quera_surface_code.zip"
    archive = str(candidate) if candidate.is_file() else None

if not (archive and vendor_dir and Path(vendor_dir).is_dir()):
    print("record data not present, section skipped")
else:
    import pymatching
    import stim

    from qb_compiler.record import RecordDem, validate
    from qb_compiler.record.dem import decode_with_weight
    from qb_compiler.record.loaders import quera_surface

    member = "Zenodo/SurfaceCodeData/Data/Distance5/d5_Z_memory.npz"
    published = quera_surface.load(archive, member, vendor_dir)
    support = validate(published).check("label_reconstruction").measured["observable_support"]

    if vendor_dir not in sys.path:
        sys.path.insert(0, vendor_dir)
    from memory import MemorySimulator
    from noise_model import NoiseModel

    n_data = 25
    simulator = MemorySimulator(
        basis="Z", noise_model=NoiseModel(NoiseModel.DEFAULT_NOISE_PARAMS), d=5, quadrant=None
    )
    circuit = stim.Circuit(
        "\n".join(str(i) for i in simulator.lc.cleanse_custom_instrs().instructions)
    )
    circuit.append("OBSERVABLE_INCLUDE", [stim.target_rec(-n_data + int(i)) for i in support], 0)
    graph = pymatching.Matching.from_detector_error_model(
        circuit.detector_error_model(decompose_errors=True, approximate_disjoint_errors=True)
    )
    n_detectors = published.n_detectors
    edges = [
        (u, n_detectors if v is None else v, a.get("weight", 1.0), bool(a.get("fault_ids")))
        for u, v, a in graph.edges()
    ]
    check = np.zeros((n_detectors, len(edges)), dtype=np.uint8)
    observable = np.zeros(len(edges), dtype=np.uint8)
    for i, (u, v, _, flips) in enumerate(edges):
        check[u, i] = 1
        if v != n_detectors:
            check[v, i] = 1
        observable[i] = 1 if flips else 0
    model = RecordDem(
        check_matrix=check,
        observable=observable,
        weights=np.asarray([e[2] for e in edges], dtype=np.float64),
    )

    with zipfile.ZipFile(archive) as handle, np.load(io.BytesIO(handle.read(member))) as payload:
        published_confidence = np.asarray(payload["gaps"], dtype=np.float64)

    prediction, weight = decode_with_weight(model, published.detector_matrix())

    full = RecordSpec(
        detectors=published.detectors,
        detector_index=published.detector_index,
        labels=published.labels,
        loss=published.loss,
        dem=model,
        meta=published.meta,
    )
    real = residual(
        full,
        {"prediction": prediction, "weight": weight},
        (published_confidence[:, None], ["published_confidence"]),
        n_nulls=20,
        seed=0,
    )
    print("QuEra d5 Z, all shots, matching decoder on the published model")
    print(f"  shots             {real.n_shots}")
    print(f"  failures          {real.n_failures}")
    print(f"  residual          {real.residual_bits:+.5f} bits per shot")
    print(f"  permutation floor {real.nulls['shot_permutation']['p95']:+.5f}")
    print(f"  pooled floor      {real.null_floor_p95:+.5f}")
    print(f"  above floor       {real.above_floor}")
    print(f"  area under curve  {real.auc_baseline:.4f} to {real.auc_augmented:.4f}")


QuEra d5 Z, all shots, matching decoder on the published model
  shots             5834
  failures          631
  residual          +0.02091 bits per shot
  permutation floor +0.00016
  pooled floor      +0.00016
  above floor       True
  area under curve  0.7952 to 0.8333


## What the number means, and what it does not

It means a feature block anticipates some of this decoder's failures that its own output did not.
That is all it means.

It is not a decoder benchmark and it does not rank decoders. A residual above the floor does not
say a better decoder exists, or that one could be built, or what it would look like. A residual at
the floor does not say the record holds nothing; it says these features on this record with this
estimator did not separate from the null.

The report states the number, the floor, both areas under the curve, the columns it used, the fold
kind, the seed and its warnings. It does not read anything into any of them, and neither should a
reader.